# GenAI Shopping Assistant — Pipeline Walkthrough

This notebook runs the actual project modules end-to-end against the real Flipkart sample dataset, so you can see each stage of the architecture (preprocessing → retrieval → ranking → comparison) produce real output. Run cells top to bottom.


## 1. Preprocessing — raw CSV → generic product schema

In [ ]:
import sys, json
sys.path.insert(0, '..')
from retrivals.preprocessing import load_flipkart, normalize_and_save

df = load_flipkart('../../datasets/flipkart.csv')  # adjust path to where you keep the raw CSV
print(df.shape)
df.head(3)


In [ ]:
summary = normalize_and_save(df)
print(json.dumps(summary, indent=2))


## 2. Retrieval — structured filter (+ semantic search if you've built a FAISS index)

In [ ]:
from backend.services.catalog import search

candidates = search(category='Mobiles', budget_max=20000, query_text='good phone under 20000')
for p in candidates[:5]:
    print(p.id, p.brand, p.title, p.price, p.rating)


## 3. Query understanding — turning free text into slots

In [ ]:
from query_understanding.intent_extraction import extract_slots

slots = extract_slots('a good phone under 20000', category_hint='Mobiles')
print(slots)


## 4. Ranking — sorting candidates by rating + price-fit + availability

In [ ]:
from ranking.rerankar import rerank

ranked = rerank(candidates, slots, top_k=5)
for p in ranked:
    print(p.id, p.brand, p.title, p.price, p.rating, p.availability)


## 5. Comparison — schema-first, grounded comparison table

In [ ]:
from comparison.comparison_engine import build_comparison

table = build_comparison(ranked[:3])
print(table['attributes'])
for row in table['rows']:
    print(row)


## 6. (Optional) Building semantic embeddings

Requires `pip install sentence-transformers faiss-cpu` and an internet connection the first time (to download the model weights). Once built, `backend/services/catalog.py` automatically blends semantic matches into `/api/query` results — no other code changes needed.


In [ ]:
# from retrivals.embeddings import build_index_for_category
# build_index_for_category('mobiles')
